# Demo en vivo · llegada de registros al pipeline

Simula eventos que llegan **uno por uno** y muestra, etapa por etapa, qué decide el pipeline.
Úsalo en la exposición: cada escenario es una celda; ejecútala y explica lo que aparece.

| Escenario | Qué se ve | Etapas |
|---|---|---|
| 1 | Una compra express real atraviesa todo el pipeline y recibe recomendaciones | 0 → 1 → 2 → 6 → 3 → 4 → 9 |
| 2 | El mismo registro llega dos veces: el segundo se descarta | 1 |
| 3 | Un registro con precio negativo se descarta | 1 |
| 4 | Seis eventos mezclados: el heap atiende primero lo urgente | 2 |
| 5 | Un usuario de la lista negra: Bloom Filter + verificación exacta | 6 |
| 6 | Un bot agrega 20 carritos en 2 minutos: Count-Min Sketch dispara la alerta | 6 |
| 7 | Tres pagos rechazados con la misma tarjeta: prueba de tarjetas | 6 |
| 8 | Autocompletado, rango de precios, más vendidos | 9 |
| 9 | Llegan 20,000 eventos más: micro-lote y la recomendación cambia | 5, 7, 8 |
| 10 | Similitud aproximada frente al método exacto (LSH vs Jaccard, HNSW vs KNN) | 7 |
| 11 | Análisis del grafo: PageRank vs grado, Louvain vs Label Propagation | 8 |

**Cómo cubre el criterio de Implementación (4 pts):**

| Lo que exige el enunciado | Dónde se ve en la demo |
|---|---|
| Un evento atraviesa las diez etapas y produce una salida observable | Escenario 1 (etapas 0→1→2→6→3→4→5/7/8→9 y recomendación) |
| Prioridades | Escenario 4 (y nivel del heap en cada evento) |
| Filtrado probabilístico | Escenarios 5, 6 y 7 (Bloom Filter y Count-Min Sketch con verificación exacta) |
| Similitud frente a un método exacto | Escenario 10 |
| Análisis del grafo | Escenario 11 (y puesto en PageRank en el escenario 1) |
| Servicio al usuario final | Escenarios 1, 8 y 9 (recomendaciones, autocompletado, Top-K, rango) |

**Tiempo:** preparar los datos y el calentamiento toma unos minutos; luego cada escenario es instantáneo.
Ejecuta las celdas 1 y 2 **antes** de la exposición.

## 1 · Preparar (ejecutar antes de exponer)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob
DRIVE = '/content/drive/MyDrive/Proyecto EDA'
CSV = f'{DRIVE}/2019-Oct.csv.zip'

if not os.path.exists('/content/proyecto/src'):
    !git clone -q https://github.com/AMPeru12/EADA.git /content/proyecto
%cd /content/proyecto
!git pull -q origin main
!pip install -q -r requirements.txt

# Datos: si ya existe un stream de pasos anteriores se reutiliza; si no, se generan 300,000 eventos (1–2 min)
if not glob.glob('data/stream_*'):
    !python src/preparar_datos.py --input "$CSV" --n 300000 --out data
STREAM = sorted(glob.glob('data/stream_*'))[0]
print('Stream:', STREAM)

In [ ]:
from src.demo import DemoEnVivo
# Calentamiento: procesa 200,000 eventos reales para que existan catálogo, índices y grafo.
# pausa = segundos entre etapas (súbelo a 0.8 si quieres que se lea con calma en la exposición)
d = DemoEnVivo(STREAM, 'data/blacklist.csv', 'data/catalogo_stock.csv', n_calentamiento=200_000, pausa=0.6)

## 2 · Escenarios
**Escenario 1 · Compra express.** Qué decir: *"Este evento es real. Entra a la fila, pasa el control de calidad,
el heap lo atiende con prioridad 1 por ser express, el guardia de fraude lo aprueba, se descuenta el stock,
se indexa por categoría y precio, y el servicio devuelve recomendaciones de cuatro estructuras distintas."*

In [ ]:
d.compra_normal()

**Escenario 2 · Duplicado.** *"Un doble clic manda el mismo registro dos veces. La segunda vez se detecta con ordenación y búsqueda binaria y se descarta."*

In [ ]:
d.duplicado()

**Escenario 3 · Registro inválido.** *"Un precio negativo no tiene sentido: se descarta y se guarda el motivo."*

In [ ]:
d.invalido()

**Escenario 4 · Prioridades.** *"Llegan seis eventos a la vez. El heap no los atiende por orden de llegada sino por urgencia, como una sala de emergencias."*

In [ ]:
d.prioridades()

**Escenario 5 · Lista negra.** *"El Bloom Filter dice 'quizás está en la lista'; antes de bloquear se confirma con la lista exacta."*

In [ ]:
d.lista_negra()

**Escenario 6 · Bot.** *"Un usuario que no está en ninguna lista agrega 20 productos en 2 minutos. El Count-Min Sketch lleva la cuenta con memoria mínima; al llegar a 18 se verifica con el conteo exacto y se alerta."*

In [ ]:
d.rafaga_bot()

**Escenario 7 · Prueba de tarjetas.** *"Cada pago rechazado genera una alerta de pago; el tercer rechazo de la misma tarjeta en 10 minutos indica que alguien está probando tarjetas robadas."*

In [ ]:
d.prueba_tarjetas()

**Escenario 8 · Servicio.** *"Lo que ve el cliente: autocompletado con el Trie, los más vendidos con Top-K y la consulta por rango de precio con el B+Tree."*

In [ ]:
d.servicio('sa')   # cambia el prefijo: 'ap', 'xi', 'le'...

**Escenario 9 · Micro-lote.** *"Llegan 20,000 eventos reales más. Las etapas caras (KD-Tree, LSH, HNSW y grafo) no se recalculan con cada clic sino por micro-lotes; al refrescarse, la recomendación del producto cambia: el sistema aprende mientras llega el stream."*

In [ ]:
d.micro_lote()

**Escenario 10 · Similitud frente al método exacto.** *"LSH responde en milisegundos lo que la fuerza bruta calcula comparando contra todos los productos, y encuentra los pares con Jaccard ≥ 0.125. HNSW recupera los mismos 10 vecinos que el KNN exacto."*

In [ ]:
d.similitud_vs_exacto()

**Escenario 11 · Grafo.** *"PageRank mide influencia y se compara con el grado; Louvain y Label Propagation encuentran grupos de productos que se compran juntos; Personalized PageRank da el cross-selling."*

In [ ]:
d.grafo()

## 3 · Tu propio evento (opcional)
Puedes inventar un registro y enviarlo. Cambia los valores y ejecuta.

In [ ]:
ev = d._plantilla('purchase')          # toma un evento real como base
ev.update({'order_type': 'express', 'price': 499.90, 'ts': d._nuevo_ts()})
d.procesar(ev, 'Mi evento')